# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [21]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants and clients
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
MODEL_LLAMA_CLOUD = 'gpt-oss:20b-cloud'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [23]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
api_key_ollama = os.getenv('OLLAMA_API_KEY')

# sanity check OPENAI keys
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# setup for OpenAI
openai = OpenAI()

# sanity check Ollama API key
if api_key_ollama and len(api_key_ollama)>10:
    print("Ollama API key looks good so far")
else:
    print("There might be a problem with your Ollama API key? Please visit the troubleshooting notebook!")

# setup for Ollama Cloud
ollama_cloud = OpenAI(base_url=OLLAMA_BASE_URL, api_key=api_key_ollama)

# setup for local Ollama
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=api_key_ollama)


API key looks good so far
Ollama API key looks good so far


In [24]:
# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

system_prompt = """
You are a clear and patient coding assistant. When given a piece of code, you will:
1. Explain what the code does, step by step.
2. Describe why the code works the way it does.
3. Provide a simple, practical example of how the code could be used.
Your explanations should be easy to follow for someone new to programming and avoid unnecessary technical jargon.
"""

user_prompt = f"""
Please explain the following code clearly and give an example of its use:
```
{question}
```
"""

In [25]:
# function to stream response and update display
def stream_response(client, model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )
    
    display_handle = display(Markdown(""), display_id=True)
    full_response = ""
    for chunk in response:
        response_text = chunk.choices[0].delta.content or ''
        full_response += response_text
        update_display(Markdown(full_response), display_id=display_handle.display_id)

In [26]:
stream_response(openai, MODEL_GPT)

Let's break down the code you provided step by step.

### What the Code Does

1. **Curly Braces (`{}`)**: The code starts with `{ ... }`, which signifies a *set comprehension*. A set is a collection that holds unique items, meaning any duplicate items will be discarded. 

2. **Expression Inside the Set Comprehension**: The part `book.get("author")` gets the value associated with the key `"author"` from each `book` in the iterable `books`. If a book does not have an author, `book.get("author")` will return `None`.

3. **Iteration Over `books`**: `for book in books` is a loop that goes through each `book` object in the `books` collection.

4. **Condition**: The phrase `if book.get("author")` ensures that only books that do have an author (meaning the author value is not `None`) are included in the set. This filters out any entries where the author key does not exist or the value is empty.

5. **`yield from`**: The `yield from` statement is used when this piece of code is part of a generator function. It allows the function to yield all values from the set created by the set comprehension. When this code is executed within a generator, it will produce each unique author one by one, rather than returning the entire set at once.

### Why the Code Works This Way

- **Uniqueness of Set**: Using a set ensures that each author is only listed once, even if multiple books have the same author. This is useful when you only want to know about distinct authors without duplicates.

- **Efficiency with Filtering**: The use of `book.get("author")` along with the condition means we only go through the list of books once. This line of code smartly combines filtering (keeping entries with authors) and collecting unique values into a single, concise statement.

- **Generator Functionality**: By using `yield from`, the code can serve in a context where you need to produce values over time, which is a memory-efficient way to handle potentially large datasets.

### Practical Example

Let’s say you have a list of books represented as dictionaries, and you want to create a generator function that yields unique authors from this list.

Here’s an example implementation:

```python
def get_unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Sample list of books
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},  # Duplicate author
    {"title": "Book 4"},                          # Missing author
    {"title": "Book 5", "author": "Author C"}
]

# Create a generator
authors_generator = get_unique_authors(books)

# Print each unique author
for author in authors_generator:
    print(author)
```

### What This Example Does:

1. It defines a function `get_unique_authors` using the code you provided.
2. It creates a list of books, some of which share the same author and one that doesn't have an author at all.
3. It creates a generator from the function and iterates over it, printing each unique author.

### Expected Output:
```
Author A
Author B
Author C
```

In this output, you can see that "Author A" is listed only once, even though it appears multiple times in the books list. The author without a name is excluded from the output.

In [27]:
stream_response(ollama, MODEL_LLAMA)

I'd be happy to explain this code in a clear and simple way.

**What the code does:**

This code is using a feature of Python called "yield" or "generator yields". It's creating a generator that will produce an iterable sequence of values, which are extracted from another list (`books`) based on certain conditions.

Here's what happens step by step:

1. The code starts by looking at each object in the `books` list (we'll call it `book`, for simplicity).
2. For each book, it checks if the "author" key exists in the book (using the `.get()` method). If it does, that means we have a valid value.
3. The code then uses a generator expression to extract the author's name from each of these books and store them in an iterable sequence (think of it as a list).
4. Finally, `yield from` is used to "forward" the extracted authors to whatever function or loop is calling this code.

**Why the code works:**

The magic here is that when we call `.get("author")` on each book object, Python performs an attribute lookup operation behind the scenes, which returns the author's name if it exists. The `yield from` statement then takes these values (which are already iterable) and combines them into a single iterable sequence.

**Example use case:**

Let's say we have this code:
```python
books = [
    {"title": "The Great Gatsby", "author": "F. Scott Fitzgerald"},
    {"title": "Pride and Prejudice", "author": "Jane Austen"},
    {"title": "War and Peace", "author": ""}  # missing author
]

authors = yield from map(lambda x: str(x["author"]), filter(None, [b.get("author") for b in books]))
```
This code creates a list of authors (`authors`), filtered to exclude any empty strings (which represent missing author information). We're using `filter()` and `map()` because they provide concise ways to process the data.

The output for this example would be: `['F. Scott Fitzgerald', 'Jane Austen']`

This code can be used in various scenarios, such as:

* Data processing pipelines for generating intermediate results.
* API response processing, where you need to yield a series of extracted values from each response object.
* Iterating over complex data structures and extracting specific parts; this generator works well for creating sequences of values.

In [28]:
stream_response(ollama_cloud, MODEL_LLAMA_CLOUD)

Below is a step‑by‑step walk‑through of the line

```python
yield from { book.get("author") for book in books if book.get("author") }
```

and a short practical example that shows how it could be used.

---

## 1. What the code does – line by line

| Step | What happens | Why it matters |
|------|--------------|----------------|
| **`book get("author") for book in books`** | This is a **comprehension** that iterates over the `books` iterable. For each `book`, it calls `book.get("author")`. |
| **`if book.get("author")`** | This part **filters** out any books that have no (truthy) `author` entry. If the value of `book.get("author")` is `None`, `''`, `0`, etc., that book is skipped. |
| **`{ ... }`** | The outer curly braces create a **set**. A set automatically removes duplicate values, so if the same author appears in more than one book, the set will contain that author only once. |
| **`yield from`** | `yield from` is used **inside a generator function**. It takes an *iterable* (here, the set created by the comprehension) and yields every element of that iterable, one by one, to the caller of the generator. Instead of writing a manual loop, `yield from` passes the control to the iterable directly. |

So, in plain English:

> *“Take every book in `books`, keep the ones that have an author, collect those authors into a set (removing duplicates), and then yield each author one at a time.”*

---

## 2. Why the code works the way it does

1. **Set comprehension**  
   Using `{ … }` instead of `[...]` means we are not building a list, we are building a *set*. Sets are a good fit here because we want unique authors and we don't care about order.  

2. **Using `.get()`**  
   `book.get("author")` protects us from a `KeyError` if a book dictionary is missing the `"author"` key. If the key is missing, `get()` returns `None`, which is falsy and therefore the `if` condition prevents that entry from being included in the set.

3. **`yield from`**  
   When you write a generator function (i.e., a function that contains at least one `yield`), you often want to delegate part of its output to another iterable. `yield from` does exactly that: it behaves like a loop `for x in iterable: yield x`, but it's shorter and, in some cases, a bit faster because the interpreter can optimize it.

4. **Memory implication**  
   Note that the set is built *before* the first `yield` happens. So the whole collection of unique authors is stored in memory. If `books` is huge and only a few authors are needed, another pattern (e.g., a generator expression) might be more memory‑friendly.

---

## 3. Simple, practical example

Let’s show a complete, runnable example that uses the line above. We'll create a small list of book dictionaries, write a generator function that yields authors, and then iterate over those authors.

```python
# Sample input: a list of book dictionaries
books = [
    {"title": "1984", "author": "George Orwell"},
    {"title": "Animal Farm", "author": "George Orwell"},
    {"title": "Brave New World", "author": "Aldous Huxley"},
    {"title": "The Great Gatsby"},                  # no author
    {"title": "To Kill a Mockingbird", "author": ""}  # empty author (falsy)
]

def unique_authors(books):
    """
    A generator that yields every unique author that appears in the
    'books' iterable. Books that lack an author are ignored.
    """
    # The line you asked about
    yield from {book.get("author") for book in books if book.get("author")}

# Use the generator
print("Authors found:")
for author in unique_authors(books):
    print(author)
```

**Output**

```
Authors found:
George Orwell
Aldous Huxley
```

### What happened?

1. The set comprehension creates `{ "George Orwell", "Aldous Huxley" }` – duplicates (`"George Orwell"`) are collapsed, and books without a truthy `"author"` (`None`, `""`, missing key) are omitted.
2. `yield from` then yields each author from that set. The loop in `unique_authors` effectively becomes:

```python
for author in {"George Orwell", "Aldous Huxley"}:
    yield author
```

---

## 4. When to use this pattern

- **When you need unique values** from an iterable and want a concise way to return them from a generator.  
- **When you want to keep the code short** but still delegate the yielding of items to an existing iterable.  
- **When the iterable you are yielding from is small enough** that building a set (or any other container) in memory is acceptable.

For very large data streams, you might prefer a generator expression instead:

```python
def unique_authors_lazy(books):
    seen = set()
    for book in books:
        author = book.get("author")
        if author and author not in seen:
            seen.add(author)
            yield author
```

This version yields authors one at a time without building the entire set up front.

---

### Bottom line

- **Line explained**: Create a set of all (unique, non‑empty) authors in `books` and yield each author.  
- **Why it works**: `yield from` passes the set’s items to the caller; the comprehension filters and deduplicates.  
- **Practical use**: Pulling out a list of distinct authors from a database of book records, where you want to expose the data via a generator.